# Day 4-01｜Roboflow Ball Detection 與 YOLO26 訓練
> Python 籃球運動資料分析課程  
> 請同學直接下載你指定的 Roboflow 籃球資料集，並準備一個 400 epochs 的 YOLO 球偵測訓練設定。  
> 修課背景：具備基礎 Python 語法即可；不預設電腦視覺或運動資料分析經驗。

## 學習目標
- 理解 ball detector 的資料集、初始化權重、訓練輸出各放在哪裡。
- 只輸入自己的 Roboflow API key，就能抓到指定 workspace / project / version 的資料集。
- 準備好一份 400 epochs 的 YOLO 訓練設定，並在你需要時自行啟動訓練。

## 完成產出
- `assets/datasets/roboflow_ball_yolo/` 你的 ball detection dataset。
- `assets/models/detectors/yolo26n_basketball_ball_best.pt` 已發布的 ball detector。
- `assets/results/d4_01_ball_detector_preview.mp4` 偵測預覽影片。

## 課堂要求
- 請同學先確認目前使用的是正確的 `uv` / `.venv` kernel。
- 若在 Google Colab，請先把執行階段切到 **T4 GPU**。
- 本 notebook 已經把訓練設定準備好，但預設不會自動開跑；請你自己決定何時執行訓練 cell。


## 執行階段提醒
- 若你是在 Google Colab 操作，請同學先把執行階段切到 **T4 GPU**，不要用純 CPU 訓練。
- 本單元會下載你指定的 Roboflow ball dataset；你只需要提供自己的 Roboflow API key。
- 初始化權重固定放在 `assets/models/pretrained/yolo26n.pt`。
- 如果訓練過程中 Ultralytics 額外在目前工作目錄產生 `yolo26n.pt`，下方 cell 會在訓練結束後自動清掉。
- 若要跑最後的 preview，請先確認 `assets/converted/` 內至少有一支 mp4。


## 課程流程

1. 確認這次要使用的 Roboflow workspace / project / version，以及模型輸出路徑。
2. 請你只輸入 API key，透過 Roboflow 官方 SDK 下載指定版本的 dataset。
3. 先檢查 YOLO detection dataset 的 splits 與類別名稱，避免拿錯資料。
4. 準備一份 `400 epochs` 的 YOLO 訓練設定，但不在這一步自動開跑。
5. 等你自行完成訓練後，再對投籃影片產生 preview。


In [2]:
from pathlib import Path
import subprocess
import sys

COURSE_ROOT_HINT = next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "course_setup.py").exists()),
    Path("/content/basketball_hackathon/course"),
)
if not (COURSE_ROOT_HINT / "src" / "course_setup.py").exists() and "google.colab" in sys.modules:
    COURSE_ROOT_HINT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "--depth", "1", "https://github.com/henry753951/basketball-hackathon-course.git", str(COURSE_ROOT_HINT)
    ], check=True)
if str(COURSE_ROOT_HINT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT_HINT))

from src.course_setup import bootstrap_course_repo  # noqa: E402

COURSE_ROOT = bootstrap_course_repo(COURSE_ROOT_HINT)


課程根目錄: H:\Repos\basketball-hackathon-course
素材資料夾: H:\Repos\basketball-hackathon-course\assets
工具模組: H:\Repos\basketball-hackathon-course\src


## Step 1｜確認資料集與模型路徑


In [3]:
from getpass import getpass

from src.roboflow_utils import dataset_status, ensure_roboflow_detection_dataset, load_yaml
from src.video_utils import display_video_in_notebook, pick_first_converted_video
from src.yolo_utils import (
    ball_detector_model_path,
    ensure_yolo26n_pretrained_weights,
    load_yolo_model,
    publish_trained_model,
    write_detection_preview_video,
    yolo26n_pretrained_path,
)

ROBOFLOW_WORKSPACE = "034-ganesh-kumar-m-v-cs-r2lwe"
ROBOFLOW_PROJECT = "basketball-lhqoe"
ROBOFLOW_VERSION = 1
ROBOFLOW_EXPORT_FORMAT = "yolov8"

BALL_DATASET_DIR = COURSE_ROOT / "assets" / "datasets" / "roboflow_ball_yolo"
BALL_MODEL_PATH = ball_detector_model_path(COURSE_ROOT)
PRETRAINED_WEIGHTS_PATH = yolo26n_pretrained_path(COURSE_ROOT)
TRAIN_PROJECT_DIR = COURSE_ROOT / "assets" / "results" / "training" / "ball_detection"
RUN_NAME = "yolo26n_basketball_ball"

print("workspace:", ROBOFLOW_WORKSPACE)
print("project:", ROBOFLOW_PROJECT)
print("version:", ROBOFLOW_VERSION)
print("dataset dir:", BALL_DATASET_DIR)
print("pretrained weights path:", PRETRAINED_WEIGHTS_PATH)
print("published model path:", BALL_MODEL_PATH)


workspace: 034-ganesh-kumar-m-v-cs-r2lwe
project: basketball-lhqoe
version: 1
dataset dir: H:\Repos\basketball-hackathon-course\assets\datasets\roboflow_ball_yolo
pretrained weights path: H:\Repos\basketball-hackathon-course\assets\models\pretrained\yolo26n.pt
published model path: H:\Repos\basketball-hackathon-course\assets\models\detectors\yolo26n_basketball_ball_best.pt


## Step 2｜只輸入 API key，下載指定的 ball dataset

這個 notebook 已經把 `workspace / project / version` 固定好了。請同學只需要提供自己的 Roboflow API key，不需要再另外手動找資料集版本。


In [4]:
ROBOFLOW_API_KEY = ""
FORCE_DOWNLOAD = False

existing_data_yaml = BALL_DATASET_DIR / "data.yaml"
if existing_data_yaml.exists() and not FORCE_DOWNLOAD:
    data_yaml = existing_data_yaml
    print("沿用既有 dataset:", data_yaml)
else:
    api_key = ROBOFLOW_API_KEY or getpass("Roboflow API key: ")
    data_yaml = ensure_roboflow_detection_dataset(
        BALL_DATASET_DIR,
        workspace=ROBOFLOW_WORKSPACE,
        project=ROBOFLOW_PROJECT,
        version=ROBOFLOW_VERSION,
        api_key=api_key,
        export_format=ROBOFLOW_EXPORT_FORMAT,
        overwrite=FORCE_DOWNLOAD,
    )
    print("downloaded data.yaml:", data_yaml)


沿用既有 dataset: H:\Repos\basketball-hackathon-course\assets\datasets\roboflow_ball_yolo\data.yaml


## Step 3｜檢查 dataset splits 與類別名稱


In [5]:
status = dataset_status(BALL_DATASET_DIR)
print(status)

if not (BALL_DATASET_DIR / "data.yaml").exists():
    raise FileNotFoundError("找不到 ball dataset 的 data.yaml。請先完成上一步下載。")

data_config = load_yaml(BALL_DATASET_DIR / "data.yaml")
print("dataset names:", data_config.get("names"))
print("train:", data_config.get("train"))
print("val:", data_config.get("val") or data_config.get("valid"))
print("test:", data_config.get("test"))


{'path': 'H:\\Repos\\basketball-hackathon-course\\assets\\datasets\\roboflow_ball_yolo', 'exists': True, 'data_yaml': 'H:\\Repos\\basketball-hackathon-course\\assets\\datasets\\roboflow_ball_yolo\\data.yaml', 'data_yaml_exists': True, 'splits': {'train': {'images': 'H:\\Repos\\basketball-hackathon-course\\assets\\datasets\\roboflow_ball_yolo\\train\\images', 'labels': 'H:\\Repos\\basketball-hackathon-course\\assets\\datasets\\roboflow_ball_yolo\\train\\labels', 'image_count': 210, 'label_count': 210}, 'val': {'images': 'H:\\Repos\\basketball-hackathon-course\\assets\\datasets\\roboflow_ball_yolo\\valid\\images', 'labels': 'H:\\Repos\\basketball-hackathon-course\\assets\\datasets\\roboflow_ball_yolo\\valid\\labels', 'image_count': 60, 'label_count': 60}, 'test': {'images': 'H:\\Repos\\basketball-hackathon-course\\assets\\datasets\\roboflow_ball_yolo\\test\\images', 'labels': 'H:\\Repos\\basketball-hackathon-course\\assets\\datasets\\roboflow_ball_yolo\\test\\labels', 'image_count': 30, 

## Step 4｜準備 YOLO26 ball detector 訓練設定

- 這裡直接使用完整資料集訓練，不再另外裁成 subset。
- 這裡已經把 `epochs` 設成 `400`，但 `RUN_TRAINING = False`，所以不會自動開跑。
- 只有在你確定要訓練時，再手動把 `RUN_TRAINING` 改成 `True`。
- 如果你是在 Colab，請確認目前執行階段仍然是 **T4 GPU**。
- 若你重跑多次，這個 cell 會優先接續最新 training run 的 `last.pt`，不會每次都從 `yolo26n.pt` 重新開始。


In [6]:
RUN_TRAINING = False
CONTINUE_FROM_LAST = True
TRAIN_CONFIG = {
    "epochs": 400,
    "imgsz": 960,
    "batch": -1,
    "workers": 4,
    "patience": 40,
    "plots": True,
    "cache": "disk",
    "close_mosaic": 12,
    "optimizer": "AdamW",
    "lr0": 0.0008,
    "cos_lr": True,
    "device": 0,
    "amp": False,
    "degrees": 0.0,
    "translate": 0.08,
    "scale": 0.35,
    "mosaic": 0.8,
    "mixup": 0.0,
    "augment": True,
}

if RUN_TRAINING:
    import hashlib

    import torch
    from ultralytics import YOLO

    print("torch:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    print("cuda version:", torch.version.cuda)
    print("device count:", torch.cuda.device_count())
    if torch.cuda.device_count():
        print("device0:", torch.cuda.get_device_name(0))
    if not torch.cuda.is_available():
        raise RuntimeError(
            "目前這個 notebook kernel 沒抓到 CUDA。請先確認 Jupyter kernel 指到 .venv，"
            "而且 torch 已經是 GPU 版，再重跑這個 cell。"
        )

    data_yaml = BALL_DATASET_DIR / "data.yaml"
    if not data_yaml.exists():
        raise FileNotFoundError(
            "找不到 data.yaml。請先完成 Roboflow ball dataset 下載。"
        )

    def file_digest(path: Path) -> str:
        digest = hashlib.sha256()
        with Path(path).open("rb") as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b""):
                digest.update(chunk)
        return digest.hexdigest()[:12]

    full_status = dataset_status(BALL_DATASET_DIR)
    print("full dataset status:", full_status)

    data_config = load_yaml(data_yaml)
    names = data_config.get("names")
    name_count = len(names) if isinstance(names, (list, dict)) else None

    init_weights = ensure_yolo26n_pretrained_weights(COURSE_ROOT)
    latest_last = None
    if CONTINUE_FROM_LAST:
        last_candidates = sorted(
            TRAIN_PROJECT_DIR.glob("*/weights/last.pt"),
            key=lambda path: path.stat().st_mtime,
            reverse=True,
        )
        if last_candidates:
            latest_last = last_candidates[0]
    start_weights = latest_last or init_weights
    train_kwargs = {
        "data": str(data_yaml),
        "project": str(TRAIN_PROJECT_DIR),
        "name": RUN_NAME,
        **TRAIN_CONFIG,
    }
    if name_count == 1:
        train_kwargs["single_cls"] = True

    print("data yaml:", data_yaml)
    print("selected init weights:", init_weights)
    print("continue from last:", CONTINUE_FROM_LAST)
    print("selected start weights:", start_weights)
    print("planned run dir:", TRAIN_PROJECT_DIR / RUN_NAME)
    print("planned published model path:", BALL_MODEL_PATH)
    print("selected start weights bytes:", start_weights.stat().st_size)
    print("selected start weights sha256[:12]:", file_digest(start_weights))
    print("train kwargs:", train_kwargs)

    model = YOLO(str(start_weights))
    results = model.train(**train_kwargs)
    save_dir = Path(getattr(getattr(model, "trainer", None), "save_dir", getattr(results, "save_dir", "")))
    trained_best = save_dir / "weights" / "best.pt"
    published_model_path = publish_trained_model(trained_best, BALL_MODEL_PATH)

    stray_weights = [Path.cwd() / "yolo26n.pt", COURSE_ROOT / "day4" / "yolo26n.pt"]
    for stray_path in stray_weights:
        if stray_path.exists() and stray_path.resolve() != init_weights.resolve():
            stray_path.unlink()
            print("removed stray init weights:", stray_path)

    print("run dir:", save_dir)
    print("trained best exists:", trained_best.exists(), trained_best)
    if trained_best.exists():
        print("published model:", published_model_path)
else:
    print("RUN_TRAINING=False；目前只準備資料與訓練設定，請你要訓練時再手動改成 True。")


RUN_TRAINING=False；目前只準備資料與訓練設定，請你要訓練時再手動改成 True。


## Step 5｜用訓練後的 ball detector 產生 preview

這一步會讀取固定發布路徑 `assets/models/detectors/yolo26n_basketball_ball_best.pt`。請你在自行完成訓練後，再回來執行這一步確認模型效果。


In [7]:
MODEL_PATH = BALL_MODEL_PATH
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        "找不到已發布的 ball detector 模型。請先把 RUN_TRAINING 改成 True 完成訓練。"
    )

model = load_yolo_model(MODEL_PATH)
model_names = getattr(model, "names", {}) or {}
class_names = [str(model_names[idx]) for idx in sorted(model_names)] if isinstance(model_names, dict) else list(model_names)
keep_class_names = [name for name in class_names if name.lower() == "ball"]
if not keep_class_names:
    keep_class_names = ["Ball", "ball", "basketball"]

video_path = pick_first_converted_video(COURSE_ROOT)
preview_path = COURSE_ROOT / "assets" / "results" / "d4_01_ball_detector_preview.mp4"
preview_path, preview_records = write_detection_preview_video(
    video_path=video_path,
    model_path=MODEL_PATH,
    output_path=preview_path,
    max_frames=120,
    conf=0.10,
    imgsz=960,
    class_names_override=class_names,
    keep_class_names=keep_class_names,
)
preview_json = preview_path.with_suffix(".json")

print("video:", video_path)
print("preview video:", preview_path)
print("preview json:", preview_json)
print("records:", len(preview_records))
display_video_in_notebook(preview_path, width=720, muted=True, loop=True)


$ C:\Users\henry\scoop\apps\ffmpeg-shared\8.1\bin\ffmpeg.exe -y -i H:\Repos\basketball-hackathon-course\assets\results\d4_01_ball_detector_preview.mp4 -an -vcodec libx264 -pix_fmt yuv420p -movflags +faststart -preset veryfast -crf 23 H:\Repos\basketball-hackathon-course\assets\results\d4_01_ball_detector_preview.notebook.mp4
video: H:\Repos\basketball-hackathon-course\assets\converted\video_001.mp4
preview video: H:\Repos\basketball-hackathon-course\assets\results\d4_01_ball_detector_preview.mp4
preview json: H:\Repos\basketball-hackathon-course\assets\results\d4_01_ball_detector_preview.json
records: 0


## Step 6｜用學生資料集接續 fine-tune

如果你想讓模型更貼近自己的課堂資料，請同學再下載 `henry753951s-workspace/basketball-ball-student-lab/1`，並沿用 Step 4 訓練完成後的發布模型繼續 fine-tune。這一步預設不會自動開始，你可以確認資料集與參數後，再把 `RUN_FINETUNE` 改成 `True`。


In [8]:
STUDENT_LAB_WORKSPACE = "henry753951s-workspace"
STUDENT_LAB_PROJECT = "basketball-ball-student-lab"
STUDENT_LAB_VERSION = 1
STUDENT_LAB_EXPORT_FORMAT = "yolov8"

STUDENT_LAB_DATASET_DIR = COURSE_ROOT / "assets" / "datasets" / "roboflow_ball_student_lab_yolo"
STUDENT_LAB_RUN_NAME = "yolo26n_basketball_ball_student_lab"
STUDENT_LAB_FORCE_DOWNLOAD = False
RUN_FINETUNE = True
FINETUNE_CONFIG = {
    "epochs": 240,
    "imgsz": 960,
    "batch": -1,
    "workers": 4,
    "patience": 60,
    "plots": True,
    "cache": "disk",
    "close_mosaic": 18,
    "optimizer": "AdamW",
    "lr0": 0.0002,
    "cos_lr": True,
    "device": 0,
    "amp": False,
    "degrees": 0.0,
    "translate": 0.03,
    "scale": 0.18,
    "mosaic": 0.15,
    "mixup": 0.0,
    "augment": True,
}

student_lab_data_yaml = STUDENT_LAB_DATASET_DIR / "data.yaml"
if student_lab_data_yaml.exists() and not STUDENT_LAB_FORCE_DOWNLOAD:
    print("沿用既有 student-lab dataset:", student_lab_data_yaml)
else:
    api_key = ROBOFLOW_API_KEY or getpass("Roboflow API key: ")
    student_lab_data_yaml = ensure_roboflow_detection_dataset(
        STUDENT_LAB_DATASET_DIR,
        workspace=STUDENT_LAB_WORKSPACE,
        project=STUDENT_LAB_PROJECT,
        version=STUDENT_LAB_VERSION,
        api_key=api_key,
        export_format=STUDENT_LAB_EXPORT_FORMAT,
        overwrite=STUDENT_LAB_FORCE_DOWNLOAD,
    )
    print("downloaded student-lab data.yaml:", student_lab_data_yaml)

student_lab_status = dataset_status(STUDENT_LAB_DATASET_DIR)
print(student_lab_status)
student_lab_config = load_yaml(student_lab_data_yaml)
student_lab_names = student_lab_config.get("names")
student_lab_name_count = len(student_lab_names) if isinstance(student_lab_names, (list, dict)) else None
print("student-lab dataset names:", student_lab_names)
print("student-lab train:", student_lab_config.get("train"))
print("student-lab val:", student_lab_config.get("val") or student_lab_config.get("valid"))
print("student-lab test:", student_lab_config.get("test"))
print("fine-tune source model:", BALL_MODEL_PATH)
print("planned fine-tune run dir:", TRAIN_PROJECT_DIR / STUDENT_LAB_RUN_NAME)
print("planned published model path:", BALL_MODEL_PATH)
print("fine-tune config:", FINETUNE_CONFIG)

if RUN_FINETUNE:
    import torch
    from ultralytics import YOLO

    if not torch.cuda.is_available():
        raise RuntimeError(
            "目前這個 notebook kernel 沒抓到 CUDA。請先確認 Jupyter kernel 指到 .venv，"
            "而且 torch 已經是 GPU 版，再重跑這個 cell。"
        )
    if not BALL_MODEL_PATH.exists():
        raise FileNotFoundError(
            "找不到 Step 4 已發布的 ball detector。請先完成前面的訓練，再回來做 fine-tune。"
        )

    finetune_kwargs = {
        "data": str(student_lab_data_yaml),
        "project": str(TRAIN_PROJECT_DIR),
        "name": STUDENT_LAB_RUN_NAME,
        **FINETUNE_CONFIG,
    }
    if student_lab_name_count == 1:
        finetune_kwargs["single_cls"] = True

    print("torch:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    print("cuda version:", torch.version.cuda)
    print("device count:", torch.cuda.device_count())
    if torch.cuda.device_count():
        print("device0:", torch.cuda.get_device_name(0))
    print("fine-tune kwargs:", finetune_kwargs)

    finetune_model = YOLO(str(BALL_MODEL_PATH))
    finetune_results = finetune_model.train(**finetune_kwargs)
    finetune_save_dir = Path(
        getattr(
            getattr(finetune_model, "trainer", None),
            "save_dir",
            getattr(finetune_results, "save_dir", ""),
        )
    )
    finetuned_best = finetune_save_dir / "weights" / "best.pt"
    finetuned_model_path = publish_trained_model(finetuned_best, BALL_MODEL_PATH)
    print("fine-tune run dir:", finetune_save_dir)
    print("finetuned best exists:", finetuned_best.exists(), finetuned_best)
    if finetuned_best.exists():
        print("published fine-tuned model:", finetuned_model_path)
else:
    print("RUN_FINETUNE=False；目前只準備 student-lab fine-tune 設定，請你要訓練時再手動改成 True。")


沿用既有 student-lab dataset: H:\Repos\basketball-hackathon-course\assets\datasets\roboflow_ball_student_lab_yolo\data.yaml
{'path': 'H:\\Repos\\basketball-hackathon-course\\assets\\datasets\\roboflow_ball_student_lab_yolo', 'exists': True, 'data_yaml': 'H:\\Repos\\basketball-hackathon-course\\assets\\datasets\\roboflow_ball_student_lab_yolo\\data.yaml', 'data_yaml_exists': True, 'splits': {'train': {'images': 'H:\\Repos\\basketball-hackathon-course\\assets\\datasets\\roboflow_ball_student_lab_yolo\\train\\images', 'labels': 'H:\\Repos\\basketball-hackathon-course\\assets\\datasets\\roboflow_ball_student_lab_yolo\\train\\labels', 'image_count': 27, 'label_count': 27}, 'val': {'images': 'H:\\Repos\\basketball-hackathon-course\\assets\\datasets\\roboflow_ball_student_lab_yolo\\valid\\images', 'labels': 'H:\\Repos\\basketball-hackathon-course\\assets\\datasets\\roboflow_ball_student_lab_yolo\\valid\\labels', 'image_count': 7, 'label_count': 7}, 'test': {'images': 'H:\\Repos\\basketball-hackat

## 本單元產出檔案

- `assets/datasets/roboflow_ball_yolo/`
- `assets/datasets/roboflow_ball_student_lab_yolo/`
- `assets/models/pretrained/yolo26n.pt`
- `assets/results/training/ball_detection/yolo26n_basketball_ball/`
- `assets/results/training/ball_detection/yolo26n_basketball_ball_student_lab/`
- `assets/models/detectors/yolo26n_basketball_ball_best.pt`
- `assets/results/d4_01_ball_detector_preview.mp4`
- `assets/results/d4_01_ball_detector_preview.json`
